# Covariance Analysis — Finviz Smallcap Universe

Objetivo: detectar si existe covarianza/correlación entre los diferentes activos del universo finviz.

**Reto de smallcaps**: los tickers rotan — no todos tienen datos simultáneos. Estrategias:
1. Filtrar por mínimo de días con datos solapados
2. Usar returns diarios (% cambio) para normalizar entre activos
3. Análisis jerárquico con clustering para detectar grupos

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_daily_cache.db'
FINVIZ_DB = '/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/trading_system_v3/data/finviz_snapshots.db'

print('Libraries loaded')

In [ ]:
# --- Cargar barras diarias ---
conn = sqlite3.connect(DB_PATH)
df_bars = pd.read_sql_query(
    'SELECT ticker, bar_date, open, high, low, close, volume FROM daily_bars ORDER BY ticker, bar_date',
    conn
)
conn.close()

df_bars['bar_date'] = pd.to_datetime(df_bars['bar_date'])
print(f"Total rows: {len(df_bars):,}")
print(f"Tickers: {df_bars['ticker'].nunique()}")
print(f"Date range: {df_bars['bar_date'].min().date()} → {df_bars['bar_date'].max().date()}")
print(f"Trading days: {df_bars['bar_date'].nunique()}")

In [ ]:
# --- Pivot: filas=fecha, columnas=ticker, valores=close ---
price_matrix = df_bars.pivot(index='bar_date', columns='ticker', values='close')

# Returns diarios (% change) — normaliza diferencias de precio entre activos
returns_matrix = price_matrix.pct_change().dropna(how='all')

print(f"Price matrix shape: {price_matrix.shape}")
print(f"Returns matrix shape: {returns_matrix.shape}")

# Cobertura de datos por ticker
coverage = returns_matrix.count()
print(f"\nDías con datos por ticker (distribución):")
print(coverage.describe().round(1))
print(f"\nTickers con >= 30 días de datos: {(coverage >= 30).sum()}")
print(f"Tickers con >= 60 días de datos: {(coverage >= 60).sum()}")
print(f"Tickers con >= 90 días de datos: {(coverage >= 90).sum()}")

In [ ]:
# --- Filtro: solo tickers con >= MIN_DAYS días de datos ---
MIN_DAYS = 30  # ajustar según necesidad
MIN_OVERLAP = 10  # días solapados mínimos entre dos tickers para calcular correlación

tickers_ok = coverage[coverage >= MIN_DAYS].index.tolist()
returns_filtered = returns_matrix[tickers_ok]

print(f"Tickers seleccionados (>= {MIN_DAYS} días): {len(tickers_ok)}")
print(f"Shape filtrado: {returns_filtered.shape}")

## 1. Matriz de Correlación

La correlación de Pearson entre returns diarios mide si dos activos se mueven juntos. Usamos `min_periods` para solo calcular cuando hay suficientes días solapados.

In [ ]:
# Matriz de correlación con mínimo de días solapados
corr_matrix = returns_filtered.corr(min_periods=MIN_OVERLAP)

# Estadísticas de correlación (excluir diagonal)
corr_values = corr_matrix.values.copy()
np.fill_diagonal(corr_values, np.nan)
flat_corr = corr_values[~np.isnan(corr_values)]

print(f"Correlación media entre pares: {np.nanmean(flat_corr):.3f}")
print(f"Correlación mediana: {np.nanmedian(flat_corr):.3f}")
print(f"Pares con |corr| > 0.5: {(np.abs(flat_corr) > 0.5).sum()} / {len(flat_corr)}")
print(f"Pares con corr > 0.7 (alta): {(flat_corr > 0.7).sum()}")
print(f"Pares con corr < -0.3 (inversa moderada): {(flat_corr < -0.3).sum()}")

In [ ]:
# Heatmap de correlación
n = len(tickers_ok)
fig_size = max(12, n * 0.35)

fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))

mask = np.isnan(corr_matrix.values)  # NaN donde no hay datos solapados

sns.heatmap(
    corr_matrix,
    ax=ax,
    mask=mask,
    cmap='RdYlGn',
    vmin=-1, vmax=1, center=0,
    square=True,
    linewidths=0.3,
    annot=(n <= 25),  # solo mostrar números si no hay muchos tickers
    fmt='.2f',
    cbar_kws={'label': 'Correlación de Pearson', 'shrink': 0.8},
    xticklabels=True,
    yticklabels=True
)

ax.set_title(f'Matriz de Correlación — Returns Diarios (universo finviz, {len(tickers_ok)} tickers, min {MIN_DAYS} días)', 
             fontsize=14, pad=15)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig('covariance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: covariance_heatmap.png')

## 2. Scatter Plot — Distribución de Correlaciones entre Pares

In [ ]:
# Extraer todos los pares únicos de correlación
tickers_list = corr_matrix.columns.tolist()
pairs = []
for i in range(len(tickers_list)):
    for j in range(i+1, len(tickers_list)):
        t1, t2 = tickers_list[i], tickers_list[j]
        c = corr_matrix.loc[t1, t2]
        if not np.isnan(c):
            # Calcular días solapados
            overlap = returns_filtered[[t1, t2]].dropna().shape[0]
            pairs.append({'ticker1': t1, 'ticker2': t2, 'correlation': c, 'overlap_days': overlap})

df_pairs = pd.DataFrame(pairs)
print(f"Pares válidos: {len(df_pairs):,}")
print(df_pairs['correlation'].describe().round(3))

In [ ]:
# --- Scatter Plot 1: Correlación vs Días Solapados ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter: correlación vs días solapados
ax1 = axes[0]
sc = ax1.scatter(
    df_pairs['overlap_days'],
    df_pairs['correlation'],
    c=df_pairs['correlation'],
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    alpha=0.5,
    s=20,
    edgecolors='none'
)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.axhline(0.5, color='green', linewidth=0.8, linestyle=':', alpha=0.7, label='corr=0.5')
ax1.axhline(-0.3, color='red', linewidth=0.8, linestyle=':', alpha=0.7, label='corr=-0.3')
plt.colorbar(sc, ax=ax1, label='Correlación')
ax1.set_xlabel('Días solapados entre par de tickers', fontsize=12)
ax1.set_ylabel('Correlación de Pearson', fontsize=12)
ax1.set_title('Correlación vs Confianza (días solapados)', fontsize=13)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Histograma de correlaciones
ax2 = axes[1]
corr_vals = df_pairs['correlation'].values
n_bins = min(50, len(corr_vals) // 10)
ax2.hist(corr_vals, bins=n_bins, color='steelblue', edgecolor='white', linewidth=0.5, alpha=0.8)
ax2.axvline(corr_vals.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Media = {corr_vals.mean():.3f}')
ax2.axvline(np.median(corr_vals), color='orange', linestyle='--', linewidth=1.5, label=f'Mediana = {np.median(corr_vals):.3f}')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Correlación de Pearson', fontsize=12)
ax2.set_ylabel('Número de pares', fontsize=12)
ax2.set_title('Distribución de Correlaciones entre Pares', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Covarianza en Universo Smallcap Finviz', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('covariance_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: covariance_scatter.png')

## 3. Scatter Plot — Pares de Tickers con Alta Correlación

In [ ]:
# Top pares más correlacionados
df_high_corr = df_pairs[df_pairs['overlap_days'] >= MIN_OVERLAP].copy()
df_high_corr = df_high_corr.sort_values('correlation', ascending=False)

print("Top 15 pares MÁS correlacionados:")
print(df_high_corr.head(15).to_string(index=False))
print("\nTop 10 pares MÁS ANTI-correlacionados:")
print(df_high_corr.tail(10).to_string(index=False))

In [ ]:
# Scatter plot de returns para los top pares más correlacionados
top_pairs = df_high_corr.head(9)

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
axes = axes.flatten()

for idx, (_, row) in enumerate(top_pairs.iterrows()):
    t1, t2 = row['ticker1'], row['ticker2']
    ax = axes[idx]
    
    # Datos solapados
    joint = returns_filtered[[t1, t2]].dropna()
    x = joint[t1].values * 100
    y = joint[t2].values * 100
    
    # Color por fecha (gradiente temporal)
    colors = plt.cm.viridis(np.linspace(0, 1, len(joint)))
    
    ax.scatter(x, y, c=colors, s=30, alpha=0.7, edgecolors='none')
    
    # Línea de regresión
    if len(x) >= 3:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        xline = np.linspace(x.min(), x.max(), 50)
        ax.plot(xline, p(xline), 'r--', linewidth=1.5, alpha=0.8)
    
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel(f'{t1} return (%)', fontsize=9)
    ax.set_ylabel(f'{t2} return (%)', fontsize=9)
    ax.set_title(f'{t1} vs {t2}\ncorr={row["correlation"]:.3f}, n={row["overlap_days"]} días', 
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Limitar outliers extremos para mejor visualización
    p5, p95 = np.percentile(np.concatenate([x, y]), [2, 98])
    margin = (p95 - p5) * 0.15
    ax.set_xlim(p5 - margin, p95 + margin)
    ax.set_ylim(p5 - margin, p95 + margin)

plt.suptitle('Top 9 Pares Más Correlacionados — Returns Diarios (color = tiempo, viridis)', 
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('top_correlated_pairs_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: top_correlated_pairs_scatter.png')

## 4. Clustering Jerárquico — Grupos de Activos Correlacionados

Con smallcaps que rotan, el clustering agrupa activos que comparten movimientos cuando coinciden en el universo.

In [ ]:
# Solo tickers con suficientes datos para clustering
MIN_DAYS_CLUSTER = 30
tickers_cluster = coverage[coverage >= MIN_DAYS_CLUSTER].index.tolist()
returns_cluster = returns_matrix[tickers_cluster]

corr_cluster = returns_cluster.corr(min_periods=MIN_OVERLAP)

# Rellenar NaN con 0 para el clustering (no correlación = independientes)
corr_filled = corr_cluster.fillna(0)

# Distancia = 1 - correlación (entre 0 y 2)
dist_matrix = 1 - corr_filled
np.fill_diagonal(dist_matrix.values, 0)  # diagonal = 0

# Clustering jerárquico
condensed = squareform(dist_matrix.values, checks=False)
linkage_matrix = linkage(condensed, method='average')

print(f"Tickers en clustering: {len(tickers_cluster)}")

In [ ]:
# Dendrograma
fig, ax = plt.subplots(figsize=(max(14, len(tickers_cluster) * 0.3), 8))

dendrogram(
    linkage_matrix,
    labels=tickers_cluster,
    ax=ax,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=0.7  # línea de corte para colorear clusters
)

ax.axhline(y=0.7, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Umbral corte (dist=0.7)')
ax.set_title(f'Clustering Jerárquico — Similitud entre Tickers Smallcap\n(distancia = 1 - correlación)', 
             fontsize=14, pad=15)
ax.set_ylabel('Distancia (1 - correlación)', fontsize=12)
ax.set_xlabel('Ticker', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('covariance_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: covariance_dendrogram.png')

## 5. Scatter Plot — ¿Existe Correlación a Nivel de Mercado?

Si los smallcaps covarían, el % de días en que la mayoría sube a la vez debería ser mayor que el azar.

In [ ]:
# Por fecha: ¿qué % de tickers subió ese día? = proxy de "estado del mercado smallcap"
up_pct_by_date = (returns_filtered > 0).mean(axis=1)  # % tickers con return positivo
mean_return_by_date = returns_filtered.mean(axis=1)  # return medio del universo
n_active = returns_filtered.count(axis=1)  # tickers activos ese día

df_market = pd.DataFrame({
    'date': up_pct_by_date.index,
    'pct_up': up_pct_by_date.values,
    'mean_return': mean_return_by_date.values,
    'n_active': n_active.values
}).dropna()

print(f"Días analizados: {len(df_market)}")
print(f"% medio de tickers en verde: {df_market['pct_up'].mean():.1%}")
print(f"Return medio diario del universo: {df_market['mean_return'].mean():.2%}")

In [ ]:
# Scatter: % tickers en verde vs return medio del universo
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter 1: % tickers verdes vs return medio
ax1 = axes[0]
sc = ax1.scatter(
    df_market['pct_up'] * 100,
    df_market['mean_return'] * 100,
    c=df_market['n_active'],
    cmap='Blues',
    s=50,
    alpha=0.7,
    edgecolors='gray',
    linewidths=0.3
)
plt.colorbar(sc, ax=ax1, label='Nº tickers activos ese día')
ax1.axhline(0, color='black', linewidth=0.8)
ax1.axvline(50, color='black', linewidth=0.8, linestyle='--', alpha=0.5)

# Regresión
x = df_market['pct_up'].values * 100
y = df_market['mean_return'].values * 100
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
xline = np.linspace(x.min(), x.max(), 50)
ax1.plot(xline, p(xline), 'r--', linewidth=2, label=f'Tendencia')

# Anotar fechas extremas
for _, row in df_market.nlargest(3, 'mean_return').iterrows():
    ax1.annotate(row['date'].strftime('%m-%d'), 
                 (row['pct_up']*100, row['mean_return']*100),
                 xytext=(5, 5), textcoords='offset points', fontsize=8, color='green')
for _, row in df_market.nsmallest(3, 'mean_return').iterrows():
    ax1.annotate(row['date'].strftime('%m-%d'), 
                 (row['pct_up']*100, row['mean_return']*100),
                 xytext=(5, -10), textcoords='offset points', fontsize=8, color='red')

ax1.set_xlabel('% Tickers en verde ese día', fontsize=12)
ax1.set_ylabel('Return medio del universo (%)', fontsize=12)
ax1.set_title('Estado del mercado smallcap por día\n(alta concentración = covarianza de mercado)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Serie temporal: evolución del % en verde
ax2 = axes[1]
df_sorted = df_market.sort_values('date')
ax2.fill_between(df_sorted['date'], df_sorted['pct_up'] * 100, 50,
                 where=df_sorted['pct_up'] > 0.5,
                 alpha=0.4, color='green', label='> 50% verdes')
ax2.fill_between(df_sorted['date'], df_sorted['pct_up'] * 100, 50,
                 where=df_sorted['pct_up'] <= 0.5,
                 alpha=0.4, color='red', label='< 50% verdes')
ax2.plot(df_sorted['date'], df_sorted['pct_up'] * 100, color='black', linewidth=1)
ax2.axhline(50, color='black', linewidth=1, linestyle='--')
ax2.set_xlabel('Fecha', fontsize=12)
ax2.set_ylabel('% Tickers en verde (%)', fontsize=12)
ax2.set_title('Evolución del estado diario del universo smallcap', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('¿Existe covarianza de mercado en el universo smallcap?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('market_covariance_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Resumen: ¿Hay Covarianza?

Interpretación de resultados:

In [ ]:
mean_corr = flat_corr.mean()
pct_positive = (flat_corr > 0).mean()
pct_high = (flat_corr > 0.5).mean()
pct_low = (flat_corr < -0.3).mean()

print("=" * 60)
print("RESUMEN: Covarianza en Universo Smallcap Finviz")
print("=" * 60)
print(f"\nCorrelación media entre todos los pares: {mean_corr:+.3f}")
print(f"% pares con correlación positiva:       {pct_positive:.1%}")
print(f"% pares con correlación alta (>0.5):    {pct_high:.1%}")
print(f"% pares con anti-correlación (<-0.3):   {pct_low:.1%}")
print()

if mean_corr > 0.2:
    print("CONCLUSIÓN: Existe covarianza POSITIVA significativa.")
    print("  Los smallcaps tienden a moverse en la misma dirección.")
    print("  Riesgo: en days de mercado bajista, las posiciones no diversifican.")
elif mean_corr > 0.05:
    print("CONCLUSIÓN: Covarianza DÉBIL positiva.")
    print("  Hay cierta correlación de mercado, pero también independencia.")
else:
    print("CONCLUSIÓN: Covarianza baja o nula.")
    print("  Los smallcaps se mueven de forma mayormente independiente.")

print()
print(f"Implicaciones para el sistema de trading:")
if mean_corr > 0.15:
    print(f"  - Evitar abrir múltiples posiciones el mismo día (riesgo correlacionado)")
    print(f"  - El estado del mercado (pct_up_day) es un filtro válido para entradas")
else:
    print(f"  - Las posiciones son mayormente independientes entre sí")
    print(f"  - El estado del mercado tiene poco impacto en las posiciones individuales")